In [1]:
import sys

sys.path.insert(0, "/tmp/kafka_ex6")

from kafka import KafkaConsumer

print("Kafka1 Python fonctionne !")

Kafka1 Python fonctionne !


In [2]:
import json
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    "weather_transformed",
    bootstrap_servers="kafka:9094",
    auto_offset_reset="earliest",
    group_id=None,
    consumer_timeout_ms=10000,
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

print("Connexion au topic weather_transformed...")

nombre = 0

for message in consumer:
    data = message.value

    print(
        "Ville :", data.get("city"),
        "| Température :", data.get("temperature"),
        "| Vent :", data.get("windspeed"),
        "| Alerte :", data.get("heat_alert_level")
    )

    nombre += 1

    if nombre >= 5:
        break

consumer.close()

print("Messages reçus :", nombre)

Connexion au topic weather_transformed...
Ville : Paris | Température : 12.7 | Vent : 1.34 | Alerte : level_0
Ville : Seville | Température : 20.3 | Vent : 4.88 | Alerte : level_0
Ville : Dubai | Température : 30.0 | Vent : 2.78 | Alerte : level_1
Ville : Reykjavik | Température : 6.6 | Vent : 6.8 | Alerte : level_0
Ville : Wellington | Température : 15.5 | Vent : 8.18 | Alerte : level_0
Messages reçus : 5


In [5]:
import json
import urllib.request

url = "http://host.docker.internal:9870/webhdfs/v1/hdfs-data?op=MKDIRS&user.name=root"

try:
    req = urllib.request.Request(
        url,
        method="PUT"
    )

    with urllib.request.urlopen(req, timeout=10) as response:
        result = json.load(response)

    if result.get("boolean"):
        print("Dossier /hdfs-data créé avec succès !")
    else:
        print("La création du dossier n'a pas été confirmée.")

except Exception as e:
    print("Erreur HDFS :", e)

Dossier /hdfs-data créé avec succès !


In [14]:
import json
import urllib.request
import urllib.error
from urllib.parse import quote, urlsplit, urlunsplit

BASE_HDFS = "http://host.docker.internal:9870/webhdfs/v1"

def url_hdfs(path, operation, options=""):
    return (
        BASE_HDFS
        + quote(path, safe="/")
        + f"?op={operation}&user.name=root&noredirect=true"
        + options
    )

def adresse_datanode(url):
    adresse = urlsplit(url)

    return urlunsplit((
        adresse.scheme,
        f"host.docker.internal:{adresse.port}",
        adresse.path,
        adresse.query,
        adresse.fragment
    ))

def lire_hdfs(path):
    url = url_hdfs(path, "OPEN")

    try:
        with urllib.request.urlopen(url, timeout=15) as response:
            resultat = json.load(response)

        destination = adresse_datanode(resultat["Location"])

        with urllib.request.urlopen(destination, timeout=15) as response:
            return response.read().decode("utf-8")

    except urllib.error.HTTPError as e:
        if e.code == 404:
            return ""
        raise

def sauvegarder_hdfs(path, contenu):
    url = url_hdfs(
        path,
        "CREATE",
        "&overwrite=true&replication=1"
    )

    requete = urllib.request.Request(url, method="PUT")

    with urllib.request.urlopen(requete, timeout=15) as response:
        resultat = json.load(response)

    destination = adresse_datanode(resultat["Location"])

    requete = urllib.request.Request(
        destination,
        data=contenu.encode("utf-8"),
        method="PUT",
        headers={"Content-Type": "application/octet-stream"}
    )

    with urllib.request.urlopen(requete, timeout=30) as response:
        print("Sauvegarde HDFS réussie :", response.status)

def ajouter_hdfs(path, nouveau_message):
    ancien_contenu = lire_hdfs(path)

    contenu_complet = ancien_contenu + nouveau_message

    sauvegarder_hdfs(path, contenu_complet)

    print("Message ajouté et ancien contenu conservé !")

print("Nouvelle méthode HDFS prête !")

Nouvelle méthode HDFS prête !


In [16]:
import json
import urllib.request
from urllib.parse import quote

HDFS_BASE = "http://host.docker.internal:9870/webhdfs/v1"

villes = {
    "France": "Paris",
    "Spain": "Seville",
    "United Arab Emirates": "Dubai",
    "Iceland": "Reykjavik",
    "New Zealand": "Wellington"
}

def creer_dossier_hdfs(path):

    url = (
        HDFS_BASE
        + quote(path, safe="/")
        + "?op=MKDIRS&user.name=root"
    )

    req = urllib.request.Request(url, method="PUT")

    with urllib.request.urlopen(req, timeout=15) as response:
        resultat = json.load(response)

    if not resultat.get("boolean"):
        raise RuntimeError("Création du dossier non confirmée : " + path)

    print("Dossier HDFS prêt :", path)


for pays, ville in villes.items():

    chemin = f"/hdfs-data/{pays}/{ville}"

    creer_dossier_hdfs(chemin)

print("\nLes cinq dossiers sont prêts !")

Dossier HDFS prêt : /hdfs-data/France/Paris
Dossier HDFS prêt : /hdfs-data/Spain/Seville
Dossier HDFS prêt : /hdfs-data/United Arab Emirates/Dubai
Dossier HDFS prêt : /hdfs-data/Iceland/Reykjavik
Dossier HDFS prêt : /hdfs-data/New Zealand/Wellington

Les cinq dossiers sont prêts !


In [19]:
import json
import time
from collections import defaultdict
from kafka import KafkaConsumer, TopicPartition

# Connexion Kafka sans réutiliser les anciens offsets
consumer_hdfs = KafkaConsumer(
    bootstrap_servers="kafka:9094",
    group_id=None,
    enable_auto_commit=False,
    value_deserializer=lambda x: json.loads(x.decode("utf-8"))
)

topic = "weather_transformed"

messages_recus = []

try:
    partitions_ids = consumer_hdfs.partitions_for_topic(topic)

    if not partitions_ids:
        raise RuntimeError("Topic Kafka introuvable")

    partitions = [
        TopicPartition(topic, p)
        for p in sorted(partitions_ids)
    ]

    consumer_hdfs.assign(partitions)

    # Reprendre au début des messages disponibles
    consumer_hdfs.seek_to_beginning(*partitions)

    print("Connexion Kafka réussie !")
    print("Lecture des messages météo...\n")

    limite = time.monotonic() + 30

    while len(messages_recus) < 10 and time.monotonic() < limite:

        messages = consumer_hdfs.poll(
            timeout_ms=2000,
            max_records=10 - len(messages_recus)
        )

        for partition, liste in messages.items():
            messages_recus.extend(liste)

finally:
    consumer_hdfs.close()


# Regrouper les messages par ville
messages_par_fichier = defaultdict(list)

for message in messages_recus:

    data = dict(message.value)

    pays = data.get("country")
    ville = data.get("city")

    if not pays or not ville:
        continue

    # Identifiants pour éviter les doublons
    data["_kafka_partition"] = message.partition
    data["_kafka_offset"] = message.offset

    chemin = f"/hdfs-data/{pays}/{ville}/alerts.json"

    messages_par_fichier[chemin].append(data)


# Enregistrer les messages dans HDFS
total_enregistres = 0

for chemin, liste_messages in messages_par_fichier.items():

    dossier = chemin.rsplit("/", 1)[0]

    creer_dossier_hdfs(dossier)

    # Lire les données déjà enregistrées
    ancien_contenu = lire_hdfs(chemin)

    offsets_existants = set()

    for ligne in ancien_contenu.splitlines():

        if ligne.strip():

            ancien_message = json.loads(ligne)

            identifiant = (
                ancien_message.get("_kafka_partition"),
                ancien_message.get("_kafka_offset")
            )

            offsets_existants.add(identifiant)

    # Éviter d'enregistrer deux fois le même message
    nouveaux_messages = []

    for data in liste_messages:

        identifiant = (
            data["_kafka_partition"],
            data["_kafka_offset"]
        )

        if identifiant not in offsets_existants:
            nouveaux_messages.append(data)

    if nouveaux_messages:

        contenu = "".join(
            json.dumps(data, ensure_ascii=False) + "\n"
            for data in nouveaux_messages
        )

        ajouter_hdfs(chemin, contenu)

        total_enregistres += len(nouveaux_messages)

        print(
            f"{len(nouveaux_messages)} messages enregistrés : {chemin}"
        )

    else:
        print("Messages déjà enregistrés :", chemin)


print("\n--------------------------------")
print("Messages Kafka lus :", len(messages_recus))
print("Nouveaux messages HDFS :", total_enregistres)
print("--------------------------------")

Connexion Kafka réussie !
Lecture des messages météo...

Dossier HDFS prêt : /hdfs-data/France/Paris
Sauvegarde HDFS réussie : 201
Message ajouté et ancien contenu conservé !
2 messages enregistrés : /hdfs-data/France/Paris/alerts.json
Dossier HDFS prêt : /hdfs-data/Spain/Seville
Sauvegarde HDFS réussie : 201
Message ajouté et ancien contenu conservé !
2 messages enregistrés : /hdfs-data/Spain/Seville/alerts.json
Dossier HDFS prêt : /hdfs-data/United Arab Emirates/Dubai
Sauvegarde HDFS réussie : 201
Message ajouté et ancien contenu conservé !
2 messages enregistrés : /hdfs-data/United Arab Emirates/Dubai/alerts.json
Dossier HDFS prêt : /hdfs-data/Iceland/Reykjavik
Sauvegarde HDFS réussie : 201
Message ajouté et ancien contenu conservé !
2 messages enregistrés : /hdfs-data/Iceland/Reykjavik/alerts.json
Dossier HDFS prêt : /hdfs-data/New Zealand/Wellington
Sauvegarde HDFS réussie : 201
Message ajouté et ancien contenu conservé !
2 messages enregistrés : /hdfs-data/New Zealand/Wellington/

In [20]:
import json

villes = {
    "France": "Paris",
    "Spain": "Seville",
    "United Arab Emirates": "Dubai",
    "Iceland": "Reykjavik",
    "New Zealand": "Wellington"
}

total = 0

for pays, ville in villes.items():

    chemin = f"/hdfs-data/{pays}/{ville}/alerts.json"

    contenu = lire_hdfs(chemin)

    lignes = [
        ligne for ligne in contenu.splitlines()
        if ligne.strip()
    ]

    nombre = len(lignes)
    total += nombre

    print(
        f"{ville} ({pays}) : {nombre} messages"
    )

print("\nTotal des messages HDFS :", total)

Paris (France) : 2 messages
Seville (Spain) : 2 messages
Dubai (United Arab Emirates) : 2 messages
Reykjavik (Iceland) : 2 messages
Wellington (New Zealand) : 2 messages

Total des messages HDFS : 10
